In [1]:
import os
import re
import pandas as pd
from PyPDF2 import PdfReader

In [2]:
def extraire_date(pdf_path):
    """Extrait la date du bulletin depuis toutes les pages du PDF"""
    with open(pdf_path, 'rb') as file:
        reader = PdfReader(file)
        date_patterns = [
            r'Bulletin du (\d{2}/\d{2}/\d{4})',
            r'Journée du (\d{2}/\d{2}/\d{4})',
            r'Date : (\d{2}/\d{2}/\d{4})'
        ]

        found_dates = []
        for page in reader.pages:
            text = page.extract_text()
            for pattern in date_patterns:
                match = re.search(pattern, text)
                if match:
                    found_dates.append((match.group(1), pattern))

        print(f"Dates trouvées dans {os.path.basename(pdf_path)} : {found_dates}")

        for date, pattern in found_dates:
            if 'Bulletin du' in pattern:
                return date.replace('/', '-')

        if found_dates:
            return found_dates[0][0].replace('/', '-')

        try:
            metadata = reader.metadata
            if metadata and '/CreationDate' in metadata:
                creation_date = metadata['/CreationDate']
                match = re.search(r'D:(\d{4})(\d{2})(\d{2})', creation_date)
                if match:
                    year, month, day = match.groups()
                    return f"{day}-{month}-{year}"
        except Exception as e:
            print(f"Erreur lors de l'extraction de la date de création : {str(e)}")

        return "date_inconnue"

In [8]:
pdf_test_path = r"C:\\Users\\zizou\\OneDrive\\Desktop\\stage 3ème\\day 2\\pdffiles\\aujourd'hui.pdf"
extraire_date(pdf_test_path)

Dates trouvées dans aujourd'hui.pdf : [('04/07/2025', 'Journée du (\\d{2}/\\d{2}/\\d{4})'), ('04/07/2025', 'Journée du (\\d{2}/\\d{2}/\\d{4})'), ('04/07/2025', 'Journée du (\\d{2}/\\d{2}/\\d{4})'), ('04/07/2025', 'Journée du (\\d{2}/\\d{2}/\\d{4})'), ('04/07/2025', 'Journée du (\\d{2}/\\d{2}/\\d{4})'), ('04/07/2025', 'Journée du (\\d{2}/\\d{2}/\\d{4})'), ('04/07/2025', 'Journée du (\\d{2}/\\d{2}/\\d{4})'), ('04/07/2025', 'Journée du (\\d{2}/\\d{2}/\\d{4})')]


'04-07-2025'

In [9]:
def extraire_tableaux(pdf_path, debut_section, fin_section):
    """Extrait les tableaux entre deux sections spécifiques"""
    with open(pdf_path, 'rb') as file:
        reader = PdfReader(file)
        texte_complet = ""
        for page in reader.pages:
            texte_complet += page.extract_text() + "\n"

        start_idx = texte_complet.find(debut_section)
        end_idx = texte_complet.find(fin_section)

        if start_idx == -1 or end_idx == -1:
            return []

        section_texte = texte_complet[start_idx + len(debut_section):end_idx]
        lignes = [ligne.strip() for ligne in section_texte.split('\n') if ligne.strip()]

        tableaux = []
        tableau_actuel = []
        for ligne in lignes:
            if re.match(r'^.*\s{2,}.*$', ligne):
                tableau_actuel.append(ligne)
            elif tableau_actuel:
                tableaux.append(tableau_actuel)
                tableau_actuel = []

        if tableau_actuel:
            tableaux.append(tableau_actuel)

        return tableaux

In [10]:
tableaux = extraire_tableaux(
    pdf_test_path,
    "Les opérations de Mise en Pension du jour",
    "Les opérations de Rétrocession des Pensions Livrées"
)
for ligne in tableaux[0]:
    print(ligne)

Montant Pension Livrée en MDT :    96,15 MDT
ISIN Libelle  Nombre de Titres  Montant  Echéance  Taux
TN0008000747         BTA 7,2% Mai 2027    3 070     3,000     265    8,000
TN2781ZB9E10         EMP NAT 2024 T1 CB TV    70 000     7,000     91    8,490
TNCYAUILJ413         BTA 9,87% 08 Janvier 2032     238    0,250     25    7,250
TNCYAUILJ413         BTA 9,87% 08 Janvier 2032    2 862     3,000     17    8,500
TNOODT102PF5         BTA 9,87% 22 Octobre 2031    28 061     30,000     31    9,440
TNCYAUILJ413         BTA 9,87% 08 Janvier 2032    1 526     1,600     31    7,750
TNN50G7PX8W5         EMP NAT 2023 T2 CB TV    86 000     8,600     17    8,600
TNVE955M6R90         EMP NAT 2023 T3 CB TF    10 000     1,000     11    8,600
TNX0K9990B08         EMP NAT 2024 T2 CB TF    29 363     3,000     10    8,500
TNX0K9990B08         EMP NAT 2024 T2 CB TF    5 873     0,600     11    8,500
TNX0K9990B08         EMP NAT 2024 T2 CB TF    5 873     0,600     11    8,500
TNX0K9990B08         EMP

In [11]:
def convertir_en_dataframe(tableau):
    """Convertit un tableau texte en DataFrame"""
    lignes_propres = []
    for ligne in tableau:
        ligne_propre = re.sub(r'\s{2,}', '|', ligne.strip())
        colonnes = ligne_propre.split('|')
        lignes_propres.append(colonnes)

    if len(lignes_propres) < 3:
        print("Tableau trop court après exclusion des lignes 0 et 1, ignoré.")
        return None

    print("Lignes propres extraites :")
    for i, ligne in enumerate(lignes_propres):
        print(f"Ligne {i}: {ligne} ({len(ligne)} colonnes)")

    header = ["ISIN", "Libellé", "Nombre de Titres", "Montant", "Échéance", "Taux"]
    lignes_normalisees = []
    for ligne in lignes_propres[2:]:
        if len(ligne) < 6:
            ligne = ligne + [''] * (6 - len(ligne))
        elif len(ligne) > 6:
            ligne = ligne[:6]
        lignes_normalisees.append(ligne)

    if not lignes_normalisees:
        print("Aucune donnée valide après exclusion des lignes 0 et 1.")
        return None

    try:
        df = pd.DataFrame(lignes_normalisees, columns=header)
        print(f"DataFrame créé avec {len(df)} lignes")
        return df
    except Exception as e:
        print(f"Erreur DataFrame : {str(e)}")
        return None

In [12]:
df = convertir_en_dataframe(tableaux[0])
df

Lignes propres extraites :
Ligne 0: ['Montant Pension Livrée en MDT :', '96,15 MDT'] (2 colonnes)
Ligne 1: ['ISIN Libelle', 'Nombre de Titres', 'Montant', 'Echéance', 'Taux'] (5 colonnes)
Ligne 2: ['TN0008000747', 'BTA 7,2% Mai 2027', '3 070', '3,000', '265', '8,000'] (6 colonnes)
Ligne 3: ['TN2781ZB9E10', 'EMP NAT 2024 T1 CB TV', '70 000', '7,000', '91', '8,490'] (6 colonnes)
Ligne 4: ['TNCYAUILJ413', 'BTA 9,87% 08 Janvier 2032', '238', '0,250', '25', '7,250'] (6 colonnes)
Ligne 5: ['TNCYAUILJ413', 'BTA 9,87% 08 Janvier 2032', '2 862', '3,000', '17', '8,500'] (6 colonnes)
Ligne 6: ['TNOODT102PF5', 'BTA 9,87% 22 Octobre 2031', '28 061', '30,000', '31', '9,440'] (6 colonnes)
Ligne 7: ['TNCYAUILJ413', 'BTA 9,87% 08 Janvier 2032', '1 526', '1,600', '31', '7,750'] (6 colonnes)
Ligne 8: ['TNN50G7PX8W5', 'EMP NAT 2023 T2 CB TV', '86 000', '8,600', '17', '8,600'] (6 colonnes)
Ligne 9: ['TNVE955M6R90', 'EMP NAT 2023 T3 CB TF', '10 000', '1,000', '11', '8,600'] (6 colonnes)
Ligne 10: ['TNX0K999

,ISIN,Libellé,Nombre de Titres,Montant,Échéance,Taux
0,TN0008000747,"BTA 7,2% Mai 2027",3 070,"3,000",265,"8,000"
1,TN2781ZB9E10,EMP NAT 2024 T1 CB TV,70 000,"7,000",91,"8,490"
2,TNCYAUILJ413,"BTA 9,87% 08 Janvier 2032",238,"0,250",25,"7,250"
3,TNCYAUILJ413,"BTA 9,87% 08 Janvier 2032",2 862,"3,000",17,"8,500"
4,TNOODT102PF5,"BTA 9,87% 22 Octobre 2031",28 061,"30,000",31,"9,440"
5,TNCYAUILJ413,"BTA 9,87% 08 Janvier 2032",1 526,"1,600",31,"7,750"
6,TNN50G7PX8W5,EMP NAT 2023 T2 CB TV,86 000,"8,600",17,"8,600"
7,TNVE955M6R90,EMP NAT 2023 T3 CB TF,10 000,"1,000",11,"8,600"
8,TNX0K9990B08,EMP NAT 2024 T2 CB TF,29 363,"3,000",10,"8,500"
9,TNX0K9990B08,EMP NAT 2024 T2 CB TF,5 873,"0,600",11,"8,500"
